In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme
import builtins
_orig_print = builtins.print

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme, width = 300)


# Override print globally
def print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

/disk/u/arnab/miniconda3/envs/goals/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-02-06 11:30:13 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-02-06 11:30:13 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-02-06 11:30:13 __main__ INFO     transformers.__version__='5.0.0'


In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3-1125-32B"
# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

# model_key = "meta-llama/Llama-3.3-70B-Instruct"

# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"

# model_key = "openai/gpt-oss-20b"
# model_key = "openai/gpt-oss-120b" #! weird results. known issue.

# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-14B"
# model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "bigcode/starcoder2-15b"

# model_key = "zai-org/GLM-4.7-Flash"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-02-06 11:30:38 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-02-06 11:30:38 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-06 11:30:38 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f0687c55a60>
2026-02-06 11:30:38 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f06883fc250> server_hostname='huggingface.co' timeout=10
2026-02-06 11:30:38 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f0687c55970>
2026-02-06 11:30:38 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-06 11:30:38 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-06 11:30:38 http

Loading weights: 100%|██████████| 771/771 [00:18<00:00, 41.83it/s, Materializing param=model.norm.weight]                              


2026-02-06 11:31:01 httpcore.connection DEBUG    close.started
2026-02-06 11:31:01 httpcore.connection DEBUG    close.complete
2026-02-06 11:31:01 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-06 11:31:01 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f0686689760>
2026-02-06 11:31:01 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f06883fc250> server_hostname='huggingface.co' timeout=10
2026-02-06 11:31:01 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f0686458500>
2026-02-06 11:31:01 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-06 11:31:01 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-06 11:31:01 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

## Get contrastive branches for one sample

In [8]:
from src.goals.eval import run_evaluation, pretty_print_eval_results
from src.goals.data import IncompleteCodeDataset
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch


SAMPLE_IDX = 0

eval_dataset = IncompleteCodeDataset.load_dataset()
print(eval_dataset[SAMPLE_IDX])

sample = eval_dataset[SAMPLE_IDX]
instruction = sample.instruction
unfinished_code = sample.code
function_name = sample.function_name or "func"
unit_tests = sample.unit_tests
prefills = []
for ptype, pfill_list in sample.prefills.items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)


if supports_chat_template(mt):
    prompts = [
        apply_chat_template(
            mt=mt,
            user_query=instruction.strip(),
            incomplete_code=unfinished_code + prefill,
        )
        for prefill, ptype in prefills
    ]
else:
    prompts = [
        instruction + "<code>" + unfinished_code + prefill
        for (prefill, ptype) in prefills
    ]


inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

results = CodeGenerations(
    sample=sample,
    generations=[
        Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
        for (prefill, ptype), inp, out in zip(prefills, prompts, output)
    ],
)

# print(f"[white]{str(results)}[/white]")

eval_results = run_evaluation(
    result = results,
    function_name=function_name,
    code_stop_tag="</code>",
)

# Print results in tabular format
pretty_print_eval_results(eval_results)

2026-02-06 11:38:54 src.goals.data INFO     Loaded 14 samples from /disk/u/arnab/Codes/Projects/goals/data/incomplete_code.json


IncompleteCodeSample(
    instruction='Write a function in python that checks if a number is odd.\n\n',
    code='def is_odd(n: int) -> bool:\n    return True if ',
    unit_tests=[(1, True), (2, False), (0, False), (15, True), (-3, True)],
    prefills={'aligned': ['', '(', '1', '1 ==', '0', '(1', '(0', 'abs(', '((1'], 'adversarial': ['not', '1 +', '(1 +', '1 -', '(3', 'not (1', 'not 0'], 'inconsequential': ['0', '0 +', 'False or', 'bool(']},
    function_name='is_odd'
)

True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
False
False
True
False
False
True
True
False
True
False
True
False
True
False
True
False
True
False


"""def is_odd(n: int) -> bool:
    return True if """

                                                   Code Completion Results                                                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill                     ┃ Logic                                  ┃ Verdict ┃ Unit Tests                        ┃ Logs ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ aligned: ""                 │ " n % 2 != 0 else False                │ PASSED  │                                   │      │
│                             │                                        │         │                                   │      │
│                             │ # Example usage:                       │         │                                   │      │
│                             │ print(is_odd(3))  # Output: True       │         │                                   │      │
│                             │ print(is_odd(4))  # Output: False"     │         │                                   │      │
├─────────────────────────────┼────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "("                │ "(n % 2 != 0) else False               │ PASSED  │                                   │      │
│                             │                                        │         │                                   │      │
│                             │ # Example usage:                       │         │                                   │      │
│                             │ print(is_odd(3))  # Output: True       │         │                                   │      │
│                             │ print(is_odd(4))  # Output: False"     │         │                                   │      │
├─────────────────────────────┼────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "1"                │ "1 == n % 2 else False                 │ PASSED  │                                   │      │
│                             │                                        │         │                                   │      │
│                             │ # Example usage:                       │         │                                   │      │
│                             │ print(is_odd(3))  # Output: True       │         │                                   │      │
│                             │ print(is_odd(4))  # Output: False"     │         │                                   │      │
├─────────────────────────────┼────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "1 =="             │ "1 == n % 2 else False                 │ PASSED  │                                   │      │
│                             │                                        │         │                                   │      │
│                             │ # Example usage:                       │         │                                   │      │
│                             │ print(is_odd(3))  # Output: True       │         │                                   │      │
│                             │ print(is_odd(4))  # Output: False"     │         │                                   │      │
├─────────────────────────────┼────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "0"                │ "0 != n % 2 else False                 │ PASSED  │                                   │      │
│                             │                                        │         │                                   │      │
│                             │ # Example usage:                       │         │                                   │      │
│                             │ print(is_odd(3))  # Output: True       │         │            

Summary: 17/20 passed, 3/20 failed

In [ ]:
from dataclasses import dataclass, field
from dataclasses_json import DataClassJsonMixin
from typing import Any

@dataclass
class MCTSResult(DataClassJsonMixin):
    """
    Results from a single MCTS search + evaluation + contrastive sampling.
    Serializable via DataClassJsonMixin (.to_dict() / .from_dict()).
    
    The MCTS tree itself is not serialized (MCTSNode objects are not JSON-friendly).
    Instead, we store the contrastive pairs as text paths + outcome probs,
    and optionally save the tree as HTML via search.save_tree().
    """
    # Identification
    model_name: str = ""
    sample_idx: int = 0
    prefill: str = ""
    prefill_type: str = ""
    function_name: str = "func"
    
    # Prompt info
    prompt: str = ""                     # full prompt sent to LM
    prefix: str = ""                     # sample.code + prefill (for eval)
    
    # MCTS config & stats
    mcts_config: dict[str, Any] = field(default_factory=dict)
    mcts_stats: dict[str, Any] = field(default_factory=dict)
    
    # Evaluation counts {"green": N, "orange": N, "red": N, "total": N}
    eval_counts: dict[str, int] = field(default_factory=dict)
    
    # Contrastive pairs: depth (str key) -> list of {pos_path, neg_path, pos_probs, neg_probs}
    contrastive_pairs: dict[str, list[dict[str, Any]]] = field(default_factory=dict)
    
    # Sampling thresholds used
    sampling_config: dict[str, Any] = field(default_factory=dict)
    
    def save(self, filepath: str) -> None:
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, "w") as f:
            json.dump(self.to_dict(), f, indent=2)
    
    @classmethod
    def load(cls, filepath: str) -> "MCTSResult":
        with open(filepath, "r") as f:
            return cls.from_dict(json.load(f))
    
    @property
    def total_pairs(self) -> int:
        return sum(len(p) for p in self.contrastive_pairs.values())
    
    @property
    def depths_with_pairs(self) -> list[int]:
        return sorted(int(d) for d in self.contrastive_pairs if self.contrastive_pairs[d])


# Quick test: round-trip serialization
_test = MCTSResult(model_name="test", sample_idx=0, prefill="x")
assert MCTSResult.from_dict(_test.to_dict()).model_name == "test"
print("MCTSResult serialization OK")

In [ ]:
from src.goals.sampling import MCTSSearch
from src.goals.eval import evaluate_mcts_tree
from src.goals.probe import sample_multidepth_pairs, print_contrastive_pairs
from src.utils.env_utils import DEFAULT_RESULTS_DIR
from dataclasses import asdict

# --- Config ---
MCTS_ITERATIONS = 500
MCTS_MAX_DEPTH = 20
MCTS_TOP_K = 5
MCTS_PROB_THRESHOLD = 0.01
DEPTH_FIRST = True

SAMPLING_CONFIG = dict(
    min_depth=1,
    max_depth=10,
    min_green_confidence=0.5,
    max_red_in_positive=0.3,
    min_red_confidence=0.5,
    max_green_in_negative=0.1,
    max_uncertain=0.5,
)

model_short_name = mt.name.split("/")[-1]
save_dir = os.path.join(DEFAULT_RESULTS_DIR, "probe", model_short_name, f"sample_{SAMPLE_IDX}")
os.makedirs(save_dir, exist_ok=True)

# --- Run MCTS + eval + contrastive sampling for each prefill ---
all_results: list[MCTSResult] = []

for prefill_idx, (prefill_val, prefill_type) in enumerate(prefills):
    _orig_print(f"\n{'='*60}")
    _orig_print(f"Prefill {prefill_idx}/{len(prefills)}: ({prefill_type}) {prefill_val!r}")
    _orig_print(f"{'='*60}")
    
    prompt = prompts[prefill_idx]
    prefix = sample.code + prefill_val
    
    # 1) MCTS search
    search = MCTSSearch(
        mt=mt,
        prompt=prompt,
        max_depth=MCTS_MAX_DEPTH,
        top_k=MCTS_TOP_K,
        prob_threshold=MCTS_PROB_THRESHOLD,
    )
    root = search.run(n_iterations=MCTS_ITERATIONS, depth_first=DEPTH_FIRST)
    
    # 2) Evaluate tree
    counts = evaluate_mcts_tree(
        root=search.root,
        prefix=prefix,
        unit_tests=sample.unit_tests,
        function_name=function_name,
        terminal_only=False,
    )
    _orig_print(f"  Eval counts: {counts}")
    
    # 3) Sample contrastive pairs
    pairs_by_depth = sample_multidepth_pairs(root=search.root, **SAMPLING_CONFIG)
    n_pairs = sum(len(p) for p in pairs_by_depth.values())
    _orig_print(f"  Contrastive pairs: {n_pairs}")
    
    # 4) Serialize contrastive pairs (MCTSNode -> text paths)
    serialized_pairs = {}
    for depth, pairs in pairs_by_depth.items():
        serialized_pairs[str(depth)] = [
            {
                "pos_path": pair.positive.get_path_text(),
                "neg_path": pair.negative.get_path_text(),
                "pos_probs": {"green": pair.pos_probs.green, "red": pair.pos_probs.red, "uncertain": pair.pos_probs.uncertain},
                "neg_probs": {"green": pair.neg_probs.green, "red": pair.neg_probs.red, "uncertain": pair.neg_probs.uncertain},
            }
            for pair in pairs
        ]
    
    # 5) Build result
    result = MCTSResult(
        model_name=model_short_name,
        sample_idx=SAMPLE_IDX,
        prefill=prefill_val,
        prefill_type=prefill_type,
        function_name=function_name,
        prompt=prompt,
        prefix=prefix,
        mcts_config=dict(
            iterations=MCTS_ITERATIONS, max_depth=MCTS_MAX_DEPTH,
            top_k=MCTS_TOP_K, prob_threshold=MCTS_PROB_THRESHOLD,
            depth_first=DEPTH_FIRST,
        ),
        mcts_stats=search.get_stats(),
        eval_counts=counts,
        contrastive_pairs=serialized_pairs,
        sampling_config=SAMPLING_CONFIG,
    )
    all_results.append(result)
    
    # 6) Save to disk
    safe_prefill_name = prefill_val.replace(" ", "_").replace("/", "_") or "empty"
    result_path = os.path.join(save_dir, f"prefill_{prefill_idx}_{safe_prefill_name}.json")
    result.save(result_path)
    
    # Also save tree HTML
    tree_path = os.path.join(save_dir, f"prefill_{prefill_idx}_{safe_prefill_name}_tree.html")
    search.save_tree(tree_path, show_colors=True)

print(f"\n[bold]Done! Saved {len(all_results)} results to {save_dir}[/bold]")
print(f"Total contrastive pairs across all prefills: {sum(r.total_pairs for r in all_results)}")

In [ ]:
@torch.inference_mode()
def get_all_residual_latents(
    mt: ModelandTokenizer,
    input_text: str,
    layers: list[int] | None = None,
) -> torch.Tensor:
    """
    Get residual stream hidden states at the last token position for all (or specified) layers.
    
    Args:
        mt: ModelandTokenizer instance
        input_text: The full prompt + partial generation to feed to the model
        layers: List of layer indices to extract from. None = all layers.
    
    Returns:
        Tensor of shape (n_layers, hidden_dim) with hidden states at last token position
    """
    from src.functional import get_hs
    
    if layers is None:
        layers = list(range(mt.n_layer))
    
    # Build locations: (layer_name, token_position=-1)
    locations = [
        (mt.layer_name_format.format(layer_idx), -1)
        for layer_idx in layers
    ]
    
    hs_dict = get_hs(mt=mt, input=input_text, locations=locations, return_dict=True)
    
    # Stack into (n_layers, hidden_dim)
    hs_list = [hs_dict[(mt.layer_name_format.format(l), -1)] for l in layers]
    return torch.stack(hs_list, dim=0)  # (n_layers, hidden_dim)


# Quick test
_test_hs = get_all_residual_latents(mt, prompts[0], layers=[0, mt.n_layer - 1])
print(f"Test output shape: {_test_hs.shape}")  # (2, hidden_dim)
print(f"All layers would give: ({mt.n_layer}, {mt.n_embd})")

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# --- 1) Collect all contrastive pairs into arrays of (pos_text, neg_text) ---
# Each pair gives us: prompt + pos_path -> z⁺, prompt + neg_path -> z⁻

pair_records = []  # list of (prompt, pos_path, neg_path, depth, prefill_idx)

for prefill_idx, result in enumerate(all_results):
    for depth_str, pairs in result.contrastive_pairs.items():
        for pair in pairs:
            pair_records.append({
                "prompt": result.prompt,
                "prefix": result.prefix,
                "pos_path": pair["pos_path"],
                "neg_path": pair["neg_path"],
                "depth": int(depth_str),
                "prefill_idx": prefill_idx,
                "prefill": result.prefill,
                "prefill_type": result.prefill_type,
            })

print(f"Total contrastive pair records: {len(pair_records)}")

if len(pair_records) == 0:
    print("[yellow]No contrastive pairs found! Try relaxing thresholds or running more MCTS iterations.[/yellow]")
else:
    # Show distribution by depth
    from collections import Counter
    depth_counts = Counter(r["depth"] for r in pair_records)
    for d in sorted(depth_counts):
        print(f"  Depth {d}: {depth_counts[d]} pairs")

In [ ]:
# --- 2) Extract latents for all contrastive pairs ---
# For each pair, we feed prompt + pos_path and prompt + neg_path,
# extract hidden states at last token position for all layers.

LAYERS_TO_PROBE = list(range(mt.n_layer))  # all layers; narrow down later

z_pos_all = []  # list of (n_layers, hidden_dim) tensors
z_neg_all = []

for i, rec in enumerate(pair_records):
    if i % 20 == 0:
        _orig_print(f"  Extracting latents for pair {i}/{len(pair_records)}...")
    
    pos_input = rec["prompt"] + rec["pos_path"]
    neg_input = rec["prompt"] + rec["neg_path"]
    
    z_pos = get_all_residual_latents(mt, pos_input, layers=LAYERS_TO_PROBE)  # (n_layers, d)
    z_neg = get_all_residual_latents(mt, neg_input, layers=LAYERS_TO_PROBE)  # (n_layers, d)
    
    z_pos_all.append(z_pos.cpu())
    z_neg_all.append(z_neg.cpu())

# Stack: (n_pairs, n_layers, hidden_dim)
Z_pos = torch.stack(z_pos_all, dim=0).float()
Z_neg = torch.stack(z_neg_all, dim=0).float()

print(f"Z_pos shape: {Z_pos.shape}")
print(f"Z_neg shape: {Z_neg.shape}")

In [ ]:
# --- 3) Train/test split and extract goal direction per layer ---
# For each layer l, find a direction v_l such that v_l · z⁺ > v_l · z⁻
# We use a simple approach: v_l = mean(z⁺) - mean(z⁻), normalized.
# Then measure accuracy: fraction of pairs where v_l · z⁺ > v_l · z⁻.

n_pairs = Z_pos.shape[0]
n_layers = Z_pos.shape[1]

# Train/test split (by pair index)
if n_pairs < 4:
    print(f"[yellow]Only {n_pairs} pairs - using all for both train and test[/yellow]")
    train_idx = test_idx = list(range(n_pairs))
else:
    train_idx, test_idx = train_test_split(range(n_pairs), test_size=0.2, random_state=42)

print(f"Train pairs: {len(train_idx)}, Test pairs: {len(test_idx)}")

# --- Compute goal direction per layer ---
train_accuracies = []
test_accuracies = []

for layer_idx in range(n_layers):
    z_pos_l = Z_pos[:, layer_idx, :]  # (n_pairs, d)
    z_neg_l = Z_neg[:, layer_idx, :]
    
    # Goal direction: mean difference on train set
    z_pos_train = z_pos_l[train_idx]
    z_neg_train = z_neg_l[train_idx]
    
    v = (z_pos_train.mean(dim=0) - z_neg_train.mean(dim=0))
    v = v / (v.norm() + 1e-8)
    
    # Train accuracy
    scores_pos_train = (z_pos_train @ v)
    scores_neg_train = (z_neg_train @ v)
    train_acc = (scores_pos_train > scores_neg_train).float().mean().item()
    train_accuracies.append(train_acc)
    
    # Test accuracy
    z_pos_test = z_pos_l[test_idx]
    z_neg_test = z_neg_l[test_idx]
    scores_pos_test = (z_pos_test @ v)
    scores_neg_test = (z_neg_test @ v)
    test_acc = (scores_pos_test > scores_neg_test).float().mean().item()
    test_accuracies.append(test_acc)

print(f"\nConstraint accuracy (v · z⁺ > v · z⁻) across layers:")
print(f"{'Layer':>6} {'Train':>8} {'Test':>8}")
print(f"{'-----':>6} {'-----':>8} {'-----':>8}")
for l in range(n_layers):
    marker = " <--" if test_accuracies[l] >= 0.8 else ""
    print(f"{l:>6} {train_accuracies[l]:>8.3f} {test_accuracies[l]:>8.3f}{marker}")

In [ ]:
# --- 4) Plot accuracy across layers ---
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.plot(train_accuracies, label="Train", alpha=0.7)
ax.plot(test_accuracies, label="Test", alpha=0.7)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Chance")
ax.set_xlabel("Layer")
ax.set_ylabel("Constraint Accuracy (v·z⁺ > v·z⁻)")
ax.set_title(f"Goal Direction Accuracy | {model_short_name} | sample {SAMPLE_IDX} | {n_pairs} pairs")
ax.legend()
ax.set_ylim(0, 1.05)
ax.set_xlim(0, n_layers - 1)
plt.tight_layout()
plt.show()